# 05 - Resultados para reporte

Objetivo: consolidar los resultados finales del proyecto para reutilizarlos en el Google Docs de entrega.

Reglas de esta etapa:
- No se entrenan modelos nuevos.
- No se modifican metricas ni split temporal.
- Solo se recopilan resultados ya generados en notebooks anteriores.
- Las tablas y conclusiones se preparan con redaccion reutilizable para el informe final.

## 1. Importacion de librerias

Se importan librerias para lectura de artefactos, resumen de tablas, visualizacion y carga del modelo guardado si fuera necesario inspeccionarlo.

In [1]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

sns.set_theme(style="whitegrid", context="notebook")

## 2. Carga de artefactos finales

Se cargan el dataset final de modelado y la tabla de metricas generada en el notebook de modelado. Estos archivos son la fuente de verdad para el resumen final.

In [2]:
DATA_PATH = Path("../data/processed/model_dataset.csv")
METRICS_PATH = Path("../reports/model_metrics.csv")
FIGURES_DIR = Path("../reports/figures")
MODEL_PATH = Path("../models/modelo_clasificacion_caida_critica.joblib")

df_model = pd.read_csv(DATA_PATH, low_memory=False, parse_dates=["fecha_data"])
model_metrics = pd.read_csv(METRICS_PATH)

print(f"Dataset final cargado: {DATA_PATH}")
print(f"Metricas cargadas: {METRICS_PATH}")
print(f"Carpeta de figuras: {FIGURES_DIR}")
print(f"Modelo guardado existe: {MODEL_PATH.exists()}")

Dataset final cargado: ..\data\processed\model_dataset.csv
Metricas cargadas: ..\reports\model_metrics.csv
Carpeta de figuras: ..\reports\figures
Modelo guardado existe: True


## 3. Dimension del dataset final

El dataset final contiene una fila por pozo-mes modelable. Incluye identificadores para trazabilidad, variables predictoras historicas y el target `caida_critica`.

In [3]:
dataset_summary = pd.DataFrame(
    {
        "metrica": [
            "filas",
            "columnas",
            "pozos_unicos",
            "fecha_minima",
            "fecha_maxima",
        ],
        "valor": [
            df_model.shape[0],
            df_model.shape[1],
            df_model["idpozo"].nunique() if "idpozo" in df_model.columns else np.nan,
            df_model["fecha_data"].min(),
            df_model["fecha_data"].max(),
        ],
    }
)

dataset_summary

,metrica,valor
0,filas,319453
1,columnas,77
2,pozos_unicos,4681
3,fecha_minima,2006-01-31 00:00:00
4,fecha_maxima,2026-03-31 00:00:00


## 4. Distribucion del target

Se resume el balance de `caida_critica`. Esta distribucion es clave para interpretar metricas, porque la clase positiva representa los eventos de caida critica futura.

In [4]:
target_distribution = (
    df_model["caida_critica"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("caida_critica")
    .reset_index(name="cantidad")
)

target_distribution["porcentaje"] = (
    target_distribution["cantidad"] / target_distribution["cantidad"].sum() * 100
).round(2)

target_distribution

,caida_critica,cantidad,porcentaje
0,0,281723,88.190
1,1,37730,11.810


## 5. Tabla comparativa de metricas

La tabla resume los modelos evaluados con el mismo split temporal. No se recalculan metricas: se leen directamente desde `../reports/model_metrics.csv`.

In [5]:
metric_cols = ["ROC AUC", "accuracy", "precision", "recall", "F1-score"]

metrics_display = model_metrics.copy()
metrics_display[metric_cols] = metrics_display[metric_cols].round(3)
metrics_display

,modelo,ROC AUC,accuracy,precision,recall,F1-score
0,RandomForestClassifier,0.667,0.656,0.196,0.553,0.290
1,LogisticRegression,0.632,0.625,0.179,0.548,0.270


In [6]:
best_model_row = model_metrics.sort_values(
    by=["ROC AUC", "recall", "F1-score"],
    ascending=False,
).iloc[0]

best_model_summary = pd.DataFrame(
    {
        "metrica": [
            "mejor_modelo",
            "ROC AUC",
            "accuracy",
            "precision",
            "recall",
            "F1-score",
        ],
        "valor": [
            best_model_row["modelo"],
            round(best_model_row["ROC AUC"], 3),
            round(best_model_row["accuracy"], 3),
            round(best_model_row["precision"], 3),
            round(best_model_row["recall"], 3),
            round(best_model_row["F1-score"], 3),
        ],
    }
)

best_model_summary

,metrica,valor
0,mejor_modelo,RandomForestClassifier
1,ROC AUC,0.667
2,accuracy,0.656
3,precision,0.196
4,recall,0.553
5,F1-score,0.290


## 6. Verificacion de figuras finales

Se verifica que existan los graficos necesarios para documentar el EDA, el modelado y la interpretabilidad del modelo elegido.

In [7]:
expected_figures = {
    "modelado_matriz_confusion": [
        "model_logreg_confusion_matrix.png",
        "model_rf_confusion_matrix.png",
    ],
    "modelado_curva_roc": [
        "model_logreg_roc_curve.png",
        "model_rf_roc_curve.png",
    ],
    "interpretabilidad": [
        "rf_feature_importance_top20.png",
    ],
    "eda_principal": [
        "eda_cobertura_mensual.png",
        "eda_produccion_mensual_total.png",
        "eda_produccion_por_provincia.png",
        "eda_produccion_por_cuenca.png",
        "eda_produccion_por_empresa.png",
        "eda_registros_por_anio.png",
        "eda_tabla_umbrales_caida.png",
        "eda_umbrales_produccion_principal.png",
    ],
    "eda_distribuciones": [
        "eda_10_hist_produccion_log1p.png",
        "eda_10_hist_produccion_p99.png",
        "eda_10_boxplot_produccion_log1p.png",
        "eda_10_ceros_por_variable.png",
        "eda_10_meses_por_pozo_hist.png",
        "eda_11_hist_var_futura_pet_pct.png",
        "eda_11_hist_var_futura_gas_pct.png",
    ],
}

figure_records = []

for categoria, archivos in expected_figures.items():
    for archivo in archivos:
        path = FIGURES_DIR / archivo
        exists = path.exists()
        figure_records.append(
            {
                "categoria": categoria,
                "archivo": archivo,
                "existe": exists,
                "tamano_kb": round(path.stat().st_size / 1024, 1) if exists else np.nan,
                "ruta_relativa": str(path),
            }
        )

figures_check = pd.DataFrame(figure_records)
figures_check

,categoria,archivo,existe,tamano_kb,ruta_relativa
0,modelado_matriz_confusion,model_logreg_confusion_matrix.png,True,25.500,..\reports\figures\model_logreg_confusion_matr...
1,modelado_matriz_confusion,model_rf_confusion_matrix.png,True,26.300,..\reports\figures\model_rf_confusion_matrix.png
2,modelado_curva_roc,model_logreg_roc_curve.png,True,51.400,..\reports\figures\model_logreg_roc_curve.png
3,modelado_curva_roc,model_rf_roc_curve.png,True,49.400,..\reports\figures\model_rf_roc_curve.png
4,interpretabilidad,rf_feature_importance_top20.png,True,83.900,..\reports\figures\rf_feature_importance_top20...
5,eda_principal,eda_cobertura_mensual.png,True,56.700,..\reports\figures\eda_cobertura_mensual.png
6,eda_principal,eda_produccion_mensual_total.png,True,179.600,..\reports\figures\eda_produccion_mensual_tota...
7,eda_principal,eda_produccion_por_provincia.png,True,40.800,..\reports\figures\eda_produccion_por_provinci...
8,eda_principal,eda_produccion_por_cuenca.png,True,39.000,..\reports\figures\eda_produccion_por_cuenca.png
9,eda_principal,eda_produccion_por_empresa.png,True,62.700,..\reports\figures\eda_produccion_por_empresa.png


In [8]:
missing_figures = figures_check.loc[~figures_check["existe"]].copy()

if missing_figures.empty:
    print("Todas las figuras esperadas existen en ../reports/figures.")
else:
    print("Faltan figuras esperadas:")
    display(missing_figures)

Todas las figuras esperadas existen en ../reports/figures.


## Resumen de resultados finales

- **Dataset final:** el dataset de modelado contiene 319.453 registros y 77 columnas. La unidad de analisis es pozo-mes y el rango temporal modelable va desde enero de 2006 hasta marzo de 2026.
- **Balance del target:** `caida_critica = 1` aparece en 37.730 registros, equivalentes al 11,81% del dataset. La clase mayoritaria `caida_critica = 0` representa el 88,19% restante.
- **Modelos evaluados:** se evaluaron `LogisticRegression` como baseline interpretable y `RandomForestClassifier` como modelo no lineal inicial, ambos con el mismo split temporal y preprocesamiento dentro de `Pipeline`.
- **Mejor modelo:** `RandomForestClassifier` fue el mejor modelo de esta etapa, con ROC AUC de 0,667, accuracy de 0,656, precision de 0,196, recall de 0,553 y F1-score de 0,290.
- **Metrica principal:** se prioriza ROC AUC para comparar la capacidad general de ranking, complementada con recall de la clase positiva porque los falsos negativos representan caidas criticas no anticipadas.
- **Interpretacion final:** el modelo es util como primera aproximacion reproducible para el curso. Aun asi, el bajo valor de precision muestra que hay muchas falsas alarmas, por lo que se recomienda seguir iterando con umbrales de decision, validacion temporal y modelos adicionales.

## Textos breves para el Google Docs

### 1. Contexto del problema

El proyecto aborda un problema de clasificacion binaria supervisada aplicado a pozos no convencionales de petroleo y gas. El objetivo es anticipar si un pozo tendra una caida critica de produccion en el mes siguiente. La unidad de analisis utilizada es pozo-mes, por lo que cada registro representa la situacion productiva de un pozo en un mes determinado. Este enfoque permite formular el problema de manera similar a un caso de alerta temprana, donde interesa detectar con anticipacion eventos de deterioro productivo.

### 2. Origen y descripcion del dataset

Se trabajo con el dataset principal de produccion mensual de pozos no convencionales, procesado finalmente en `model_dataset.csv`. El dataset final contiene 319.453 registros y 77 columnas, con informacion productiva, temporal, geografica, tecnica y categorica de los pozos. El rango temporal modelable va desde enero de 2006 hasta marzo de 2026, e incluye 4.681 pozos unicos. Para el modelado se conservaron variables utiles para prediccion y trazabilidad, evitando incluir columnas futuras o auxiliares que pudieran introducir leakage.

### 3. Definicion del target `caida_critica`

La variable objetivo `caida_critica` fue definida como un evento binario. Toma valor 1 cuando la produccion principal del pozo cae un 30% o mas en el mes siguiente, y toma valor 0 cuando esa caida futura no alcanza dicho umbral. La produccion principal se definio segun el tipo de pozo: para pozos petroliferos se uso `prod_pet`, y para pozos gasiferos se uso `prod_gas`. Para evitar inconsistencias temporales, el target se calculo solo cuando el siguiente registro observado correspondia exactamente al mes calendario siguiente.

### 4. Resumen del EDA

El analisis exploratorio mostro una fuerte concentracion geografica y productiva en Neuquen y la Cuenca Neuquina, ademas de diferencias claras entre pozos gasiferos y petroliferos. Las variables de produccion presentaron distribuciones muy sesgadas, con outliers relevantes y presencia de ceros. Tambien se observaron pozos con historiales cortos y otros con historiales extensos. Estos hallazgos justificaron usar transformaciones visuales como escala log1p en el EDA, analizar umbrales de caida y construir el target sobre la produccion principal de cada pozo.

### 5. Preprocesamiento y feature engineering

El feature engineering se realizo respetando la temporalidad del problema. Se crearon variables temporales, antiguedad del pozo, ratios productivos, lags, variaciones historicas, medias moviles, desvios moviles, acumulados y flags de produccion. Todas estas variables usan informacion del mes actual o de meses anteriores, nunca del mes siguiente. Las variables futuras utilizadas para construir el target, como la produccion del mes siguiente o la variacion futura, fueron excluidas del dataset de modelado para evitar data leakage.

### 6. Modelos utilizados

Se evaluaron dos modelos con el mismo split temporal y el mismo conjunto de features. Primero se entreno una `LogisticRegression` como baseline interpretable. Luego se entreno un `RandomForestClassifier` con parametros conservadores para capturar relaciones no lineales sin aumentar excesivamente el riesgo de overfitting. En ambos casos, el preprocesamiento se realizo dentro de un `Pipeline`, incluyendo imputacion, escalado para variables numericas cuando correspondia y codificacion One Hot para variables categoricas.

### 7. Evaluacion de resultados

La evaluacion se realizo con split temporal, usando los meses mas antiguos para entrenamiento y los meses mas recientes para test. Este criterio es adecuado porque el objetivo es predecir eventos futuros y no mezclar informacion temporalmente posterior en el entrenamiento. El mejor modelo fue `RandomForestClassifier`, con ROC AUC de 0,667, accuracy de 0,656, precision de 0,196, recall de 0,553 y F1-score de 0,290. La regresion logistica obtuvo ROC AUC de 0,632, accuracy de 0,625, precision de 0,179, recall de 0,548 y F1-score de 0,270.

### 8. Interpretacion de errores

En este problema, los falsos negativos son especialmente importantes: representan pozos-mes que efectivamente tuvieron una caida critica, pero que el modelo clasifico como no criticos. En una aplicacion operativa, estos casos implicarian no generar una alerta ante una caida futura de produccion. Por este motivo, el recall de la clase positiva es una metrica central. El Random Forest alcanzo un recall de 0,553 para la clase positiva, detectando algo mas de la mitad de los eventos criticos, aunque con precision baja, lo que indica presencia de falsas alarmas.

### 9. Limitaciones

El proyecto presenta algunas limitaciones. En primer lugar, la clase positiva esta desbalanceada: `caida_critica = 1` representa 37.730 casos, equivalentes al 11,81% del dataset final. En segundo lugar, el modelo todavia presenta precision baja para la clase positiva, por lo que muchas alertas pueden corresponder a falsos positivos. Ademas, la importancia de variables del Random Forest permite interpretar senales predictivas, pero no implica causalidad. Por ultimo, la definicion del target depende del umbral seleccionado del 30%, que deberia validarse con criterio tecnico y de negocio.

### 10. Recomendaciones futuras

Como proximos pasos, se recomienda evaluar otros modelos no lineales, ajustar el umbral de decision segun el costo relativo de falsos negativos y falsos positivos, y aplicar validacion temporal por ventanas. Tambien seria conveniente analizar la importancia de variables con metodos complementarios, revisar segmentos especificos por tipo de pozo o cuenca, y validar la definicion de caida critica con conocimiento experto del dominio. Finalmente, para una version productiva, el pipeline deberia monitorearse en el tiempo para detectar cambios en la distribucion de datos y perdida de desempeno.